In [14]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import display, Markdown
import gradio as gr
from pypdf import PdfReader

In [15]:
load_dotenv(override = True)
openai = OpenAI()

In [16]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text
print(linkedin)

   
Contact
kavyahj26@gmail.com
www.linkedin.com/in/kavya-h-j
(LinkedIn)
my-porttfolio-lemon.vercel.app
(Personal)
Top Skills
Motivation - Daily quotes
AI Analytics
Data Engineering
Certifications
Certificate+of+Completion:+Al
+Fluency+Framework+
Google AI Essentials
SQL (Intermediate)
Certificate of completion:
Introduction to Claude Cowork
Python (Basic) Certificate
Kavya Hosamane Jayanna
Full Stack AI Engineer | RAG · LLM · .NET · React · AWS | MS CS @
Binghamton ’26
Greater Binghamton
Summary
Graduated with a Master's degree in Computer Science from
Binghamton University and a Bachelor's degree in Electronics and
Communication from MVJ College of Engineering, this professional
is AWS Certified Cloud Practitioner with expertise in Amazon EC2,
AWS Identity and Access Management (IAM), and Git.  
At Happiest Minds Technologies, contributed as a Senior Software
Engineer, focusing on developing innovative solutions and
leveraging cloud technologies. Dedicated to creating scalable
system

In [18]:
with open("twin/summary.txt", "r", encoding="utf8-") as f:
    summary = f.read()
print(summary)

My name is Kavya Hosamane Jayanna. 
I'm an AI Engineer and software engineer. I'm originally from India, but I moved to New York in 2025.
I love all foods, particularly Italian food, I also like Indian food because of its spiceness and flavour but I really hate Cookies and Breads, Idk how people like them, but I definettly dont like the taste of it. 



In [17]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

If they directly want to speak to me ask their name, email and phone number to record it

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [19]:
def chat(message, history):
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages)
    return response.choices[0].message.content

In [7]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
def record_user_details_tool(email, name, phone):
    print(f"Tool called to record an email:{email}")
    with open("user_details.txt", "a", encoding="utf-8") as f:
        line = name + "-" + email
        if phone:
            line += f" ({phone})"
        f.write(line + "\n")
    return "user details received!!"

In [21]:
record_user_details_tool_json = {
    "name": "record_user_details_tool",
    "description": "Use this tool to record that a user provided their contact details",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The name of this user"},
            "phone_number": {"type": "string", "description": "The phone number of this user, if they provided it, else an empty string "}
        },
        "required": ["email", "name"],
        "additionalProperties": False
    }
}

In [22]:
tools = [{"type": "function", "function": record_user_details_tool_json}]


In [25]:
def chat(message, history):
    messages = [{"role":"system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model = "gpt-5.4-mini", messages = messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email  = json.loads(tool_call.function.arguments).get("email")
            user = json.loads(tool_call.function.arguments).get("name")
            phone_number = json.loads(tool_call.function.arguments).get("phone_number")
            record_user_details_tool(email, user, phone_number)
            messages.append({"role":"tool","content": "user details recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [29]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
